testing HDF5 conversion to other formats

In [1]:
# import pandas as pd
import h5py
import os
from fod_config import *


In [2]:
h5name = "narr_PSD_2008_BC"
# h5folder = "../../h5data"
h5folder = NARR_INPUT_DIR 
h5filename = h5folder + "/" + h5name + ".h5"
os.path.exists(h5filename)



True

In [3]:
hd_key = 'PC'
dataset_name = 'PC' 
# csvfile =  h5folder + "/" + h5name +  "_" + dataset_name + ".csv"

In [4]:
with h5py.File(h5filename,'r') as hf:    
    print('List of arrays in this file: \n', list(hf.keys()))

List of arrays in this file: 
 ['PC', 'WD', 'WS']


**Work with our new FOD module**

In [5]:
from fod3 import read_narr_lat_lon,validate_latlon,read_one_year,read_narr_timeseries

MIOFFSET DEVELOPMENT VERSION ONLY - NOT FOR PRODUCTION USE


In [6]:
yr = 2008
idy = 1
idx = 1
pc_1year, ws_1year, wd_1year = read_one_year(yr=yr, idy=idy, idx=idx, narr_input_dir=h5folder)
ws_1year

array([6.277, 5.044, 5.461, ..., 5.236, 5.085, 5.546], shape=(2920,))

In [7]:
def read_one_year_grid(yr:str,narr_input_dir:str):
    """read one year hf5 file, extra 3 datasets
    Files must be named like narr_PSD_1980_BC.h5
    
    Args:
        yr (int): year of data to read, embedded in filename
        idy (int): index of grid y coordinate (North/South)
        idx (int): index of grid x coordinate (East/West)
        narr_input_dir (str): path to NARR input files
    Returns:
        tuple of np arrays: timeseries values for PC, WD and WS from one grid point, all hours
    """
    narr_file_name = f"narr_PSD_{yr}_BC.h5"
    
    h5f_annual_filename = os.path.join(narr_input_dir, narr_file_name)
    h5f = h5py.File(h5f_annual_filename, 'r')
    # extract all values for one year
    # previously filtered at read time, like
    #  pc_1year = h5f['pc'][idy,idx,ts:te]
    pc_1year = h5f['PC']
    ws_1year = h5f['WS']
    wd_1year = h5f['WD']
    return pc_1year, ws_1year, wd_1year

In [8]:
pc_1year, ws_1year, wd_1year = read_one_year_grid(yr=yr, narr_input_dir=h5folder)


In [9]:
dir(pc_1year)
pc_1year.shape


(277, 349, 2920)

In [10]:
pc_1year.shape[0] * pc_1year.shape[1]

96673

size of ALL data

In [11]:
start_year = 1979; end_year = 2009
years = range(start_year, end_year+1)  # range 'stop' is the value after the top value becuse PYTHON
count_all_vals = (years[-1] - years[0]) * (pc_1year.shape[0] * pc_1year.shape[1]) * pc_1year.shape[2]

print( "number of vals in 30 yr  stack" , count_all_vals )
print( "gb" , (count_all_vals * 4)/ (1024**3) )

number of vals in 30 yr  stack 8468554800
gb 31.547825038433075


if we have the memory it is less IO to read everythign into memory but need 

In [12]:
print( "number of vals in 30 yr point stack" ,  (years[-1] - years[0]) * (pc_1year.shape[0] * pc_1year.shape[1]))

number of vals in 30 yr point stack 2900190


Psuedo code to export this to JSON. 

Note that the fod main model currently assumes there are 2920 values per year and just smushes them all together in a long time series with no time associated with them
in the JSON file should have years for each, and a process for doing the smushing

1. read in all years in 3 giant 3d matrices.   or naively 3 dictionaries with year

PC = {1979: 3d np.array, 1980: 3dnp.array. etc}
WD = etc

matrix of x,ys   or list of x,y combinations from one of the years

2. grid_points 
shape is (277, 349, 2920)
so grid points are x: (0:276) y = rang(0:348) pc_1year.shape[0] * pc_1year.shape[1] -> 96673 points

if we save by grid point that is 96K files.  

each file will be 

3. loop for each grid point x,y
point_vals = {}
for each year
    point_vals[year] = get values for x,y

4. store in AWS: 
best practices https://reintech.io/blog/organizing-data-amazon-s3-best-practices

single bucket, but unique indices depending on how data is read

object name has elements Variable (PC, WD, WS) x, y
based on above better to use x,y first and var name last

`{x}/{y}/PC.json`   <- all years in this file for this coordinate

- program needs to find only 3 object, PC, WD, WS
- each object has 2920 timepoint * 30 yrs = 

single bucket means single point of permisions setting since all data will be
read by same program

```python
# setup aws clie config and IAM user creds in folder
import boto3
# global is easier for this one-off
client = boto3.client('s3')
bucket_name  = "narr_wind_timeseries"


def saveS3(winddata, varname,x,y):
    object_name = f"{x}/{y}/{varname}_{x}_{y}.json"
    d = json.dumps(winddata)
    try:
        res = client.put_object(Body=d, Bucket=bucket_name, Key=object_name)
    except exception as e:
        print(f"can't write {object_name} to bucket")
        raise
    print(res)
```

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv(override = True)
print(os.getenv("REGION_NAME"))
print(os.getenv("NARR_INPUT_DIR"))





us-east-1
/mnt/research/ICER-RSE/clients/enviroweather/mioffset/h5


create an AWS session, but ensure to use the AWS config that's in the .env fle or set by the program instead of the default config that may be set in your `~/.aws/config` file

In [2]:
aws_config = { 
    "aws_access_key_id" : os.getenv("AWS_ACCESS_KEY_ID"), 
    "aws_secret_access_key" : os.getenv("AWS_SECRET_ACCESS_KEY"), 
    "region_name" : os.getenv("REGION_NAME") 
}

import boto3
session = boto3.Session(**aws_config)
session

Session(region_name='us-east-1')

In [3]:
s3_client = session.client('s3')

# List all buckets
response = s3_client.list_buckets()
print([bucket['Name'] for bucket in response['Buckets']])

bucket_list = [ bucket['Name'] for bucket in list(response['Buckets'])]
print(os.getenv('BUCKET_NAME') in bucket_list)

['elasticbeanstalk-us-east-1-941061635635', 'mioffsetnarr']
True


just for sharing purposes, put all the NARR files into that bucket as they are

In [11]:
def path_to_narrfile(yr:int, narr_input_dir):
    narr_file_name = f"narr_PSD_{yr}_BC.h5"
    h5f_annual_filename = os.path.join(narr_input_dir, narr_file_name)
    return(h5f_annual_filename)

narr_input_dir = os.getenv('NARR_INPUT_DIR')

yr = 1980
p = path_to_narrfile(yr,narr_input_dir)
print(p)
print(os.path.exists(p))

/mnt/research/ICER-RSE/clients/enviroweather/mioffset/h5/narr_PSD_1980_BC.h5
True
